# LilyVLM SFT Dataset Preprocessing & Packaging

This notebook downloads, cleans, and packages the Supervised Fine-Tuning (SFT) dataset for **LilyVLM**. It creates a unified dataset by mixing:
1. **48,000 Multimodal Reasoning Samples** from `MMR1/MMR1-SFT` (Partitions 0 to 5).
2. **16,000 Filtered Multi-Domain Text Reasoning Samples** from `abhinav0231/Sarvam-105b-Distill-100k` (Filtered to remove incomplete context prompts like BCCI agreements).

The final mixed dataset is pushed directly to your Hugging Face account as a single, ready-to-use dataset: `abhinav0231/Lily-SFT-Clean-Dataset`.

In [1]:
# Install requirements
!pip install -q datasets huggingface_hub pillow tqdm

In [2]:
# ==============================================================================
# Cell 2 — Imports, Authentication & Global Configuration
# ==============================================================================
import os, glob, tarfile, json, random, re
from PIL import Image
import requests
from datasets import Dataset, Features, Image as DImage, Value, load_dataset
try:
    from huggingface_hub import hf_hub_download, login, get_token
except ImportError:
    from huggingface_hub import hf_hub_download, login, HfFolder
    get_token = HfFolder.get_token

# Retrieve HF Token
HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN", "")
if not HF_TOKEN or HF_TOKEN == "YOUR_HF_TOKEN_HERE":
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN") or userdata.get("HUGGINGFACE_TOKEN") or ""
    except Exception:
        pass

if not HF_TOKEN or HF_TOKEN == "YOUR_HF_TOKEN_HERE":
    cached_token = get_token()
    if cached_token:
        HF_TOKEN = cached_token

if HF_TOKEN and HF_TOKEN != "YOUR_HF_TOKEN_HERE":
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    try:
        login(token=HF_TOKEN, add_to_git_credential=False)
        print("✅ Authenticated with Hugging Face")
    except Exception as e:
        print(f"⚠️ Hugging Face authentication note: {e}")
else:
    print("ℹ️ HF_TOKEN not provided. Proceeding (public datasets/models remain accessible).")

HF_USERNAME = "abhinav0231"
TARGET_REPO = f"{HF_USERNAME}/Lily-Vision-SFT-Dataset"
SEED = 42
print(f"Target repository: {TARGET_REPO}")


Target repository: abhinav0231/Lily-Vision-SFT-Dataset


In [3]:
# 2. Download and Extract MMR1-SFT Partitions
extracted_dir = "./extracted_mmr1_data"
os.makedirs(extracted_dir, exist_ok=True)
dataset_id = "MMR1/MMR1-SFT"
num_partitions = 6
print(f">> Downloading and extracting {num_partitions} partitions of MMR1-SFT using standard hf_hub_download...")
for i in range(num_partitions):
    filename = f"part_{i:04d}.tar"
    print(f"\n--- Loading {filename} ---")
    try:
        # Official library downloader (redirected to hf-mirror)
        local_tar_path = hf_hub_download(
            repo_id=dataset_id,
            repo_type="dataset",
            filename=filename,
            token=HF_TOKEN,
            local_dir=extracted_dir
        )

        print(f"Downloaded to {local_tar_path}. Extracting...")
        with tarfile.open(local_tar_path, "r") as tar:
            tar.extractall(path=extracted_dir)
        print(f"Extracted {filename} successfully.")

        # Clean up local tar to save space
        if os.path.exists(local_tar_path):
            os.remove(local_tar_path)
    except Exception as e:
        print(f"ERROR loading partition {i}: {e}")

>> Downloading and extracting 6 partitions of MMR1-SFT using standard hf_hub_download...

--- Loading part_0000.tar ---


part_0000.tar: reconstructing file:   0%|          |  0.00B / 4.28GB            

part_0000.tar: downloading bytes:           |  0.00B            

Downloaded to /content/extracted_mmr1_data/part_0000.tar. Extracting...


/tmp/ipykernel_1311/4109106767.py:22: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=extracted_dir)


...

part_0004.tar: downloading bytes:           |  0.00B            

Downloaded to /content/extracted_mmr1_data/part_0004.tar. Extracting...
Extracted part_0004.tar successfully.

--- Loading part_0005.tar ---


part_0005.tar: reconstructing file:   0%|          |  0.00B / 4.31GB            

part_0005.tar: downloading bytes:           |  0.00B            

Downloaded to /content/extracted_mmr1_data/part_0005.tar. Extracting...
Extracted part_0005.tar successfully.


In [4]:
# 3. Gather and format Visual Samples
multimodal_samples = []

print(">> Processing extracted visual samples...")

# 1. Gather DocReason samples
docreason_file = "./extracted_mmr1_data/DocReason/annotations.json"
if os.path.exists(docreason_file):
    print("Loading DocReason dataset...")
    with open(docreason_file, "r", encoding="utf-8") as f:
        doc_data = json.load(f)
    print(f"Loaded {len(doc_data)} raw DocReason samples.")

    for item in doc_data:
        relative_path = item["relative_image_path"]
        # Clean path formatting if needed
        if relative_path.startswith("./"):
            relative_path = relative_path[2:]
        img_path = os.path.join("./extracted_mmr1_data/DocReason", relative_path)

        if os.path.exists(img_path):
            question = item["question"]
            thought = item["CoT"]
            answer = item["final_answer"]

            full_response = f"<think>\n{thought}\n</think>\n<answer>\n{answer}\n</answer>"
            multimodal_samples.append({
                "image": img_path,
                "prompt": f"<|im_start|>system\nYou are a helpful assistant. Always reason step-by-step inside <think></think> tags, then write your final answer inside <answer></answer> tags.<|im_end|>\n<|im_start|>user\n<image>\n{question}<|im_end|>\n<|im_start|>assistant\n",
                "response": full_response
            })
    print(f"Loaded {len(multimodal_samples)} valid DocReason samples with existing images.")
else:
    print("DocReason annotations file not found!")

# 2. Gather AtomMath-sft samples to reach target 48,000 total
atommath_file = "./extracted_mmr1_data/AtomMath-sft/AtomMATH-SFT.json"
if os.path.exists(atommath_file):
    print("Loading AtomMath dataset...")
    with open(atommath_file, "r", encoding="utf-8") as f:
        atom_data = json.load(f)
    print(f"Loaded {len(atom_data)} raw AtomMath samples.")

    # Shuffle using dataset SEED to ensure random selection of math domains
    random.seed(SEED)
    random.shuffle(atom_data)

    loaded_atom = 0
    target_atom = 48000 - len(multimodal_samples)
    print(f"Targeting {target_atom} samples from AtomMath...")

    for item in atom_data:
        if loaded_atom >= target_atom:
            break

        relative_path = item["image"]
        img_path = os.path.join("./extracted_mmr1_data/AtomMath-sft/images", relative_path)

        if os.path.exists(img_path):
            convs = item["conversations"]
            human_val = convs[0]["value"]
            gpt_val = convs[1]["value"]

            # Parse question
            q_match = re.search(r"THE GIVEN QUESTION:\n(.*?)\nAnswer the question", human_val, re.DOTALL)
            question = q_match.group(1).strip() if q_match else human_val.strip()
            question = question.replace("<image>\n", "").replace("<image>", "").strip()

            # Parse historical reasoning steps
            steps_match = re.search(r"HISTORICAL REASONING STEPS:\n(.*)", human_val, re.DOTALL)
            historical_steps = steps_match.group(1).strip() if steps_match else ""

            # Combine reasoning steps for thought (with meta-prompt cleaning)
            thought = f"{historical_steps}\n{gpt_val}".strip()
            thought = re.sub(r"Your task is to predict the next step.*", "", thought, flags=re.IGNORECASE | re.DOTALL).strip()
            thought = re.sub(r"If the historical reasoning steps.*", "", thought, flags=re.IGNORECASE | re.DOTALL).strip()
            thought = re.sub(r"THE GIVEN QUESTION:\n.*", "", thought, flags=re.IGNORECASE | re.DOTALL).strip()
            thought = thought.strip()

            # Parse answer
            ans_match = re.search(r"the final answer is:\s*(.*)", gpt_val, re.IGNORECASE)
            answer = ans_match.group(1).rstrip(".").strip() if ans_match else gpt_val.strip()

            full_response = f"<think>\n{thought}\n</think>\n<answer>\n{answer}\n</answer>"
            multimodal_samples.append({
                "image": img_path,
                "prompt": f"<|im_start|>system\nYou are a helpful assistant. Always reason step-by-step inside <think></think> tags, then write your final answer inside <answer></answer> tags.<|im_end|>\n<|im_start|>user\n<image>\n{question}<|im_end|>\n<|im_start|>assistant\n",
                "response": full_response
            })
            loaded_atom += 1

    print(f"Loaded {loaded_atom} valid AtomMath samples. Total visual samples: {len(multimodal_samples)}.")
else:
    print("AtomMath annotations file not found!")

if len(multimodal_samples) == 0:
    raise ValueError(
        "CRITICAL ERROR: No visual samples were loaded! "
        "Please check the extracted files directory to verify extraction paths."
    )

>> Processing extracted visual samples...
Loading DocReason dataset...
Loaded 21797 raw DocReason samples.
Loaded 21797 valid DocReason samples with existing images.
Loading AtomMath dataset...
Loaded 157449 raw AtomMath samples.
Targeting 26203 samples from AtomMath...
Loaded 26203 valid AtomMath samples. Total visual samples: 48000.


In [5]:
# 4. Load & Programmatically Clean Multi-Domain Text Reasoning Samples
def clean_sarvam_answer(answer):
    answer = answer.strip()
    final_idx = answer.lower().find("<final>")
    if final_idx != -1:
        result = answer[final_idx + 7:].strip()
        close_final_idx = result.lower().find("</final>")
        if close_final_idx != -1:
            result = result[:close_final_idx].strip()
        result = re.sub(r"</?(?:thinking|think|final|tool_call)[^>]*>", "", result, flags=re.IGNORECASE)
        return result.strip()
    last_close_idx = -1
    for tag in ["</thinking>", "</think>", "</tool_call>"]:
        idx = answer.lower().rfind(tag)
        if idx != -1:
            last_close_idx = max(last_close_idx, idx + len(tag))
    if last_close_idx != -1:
        result = answer[last_close_idx:].strip()
        result = re.sub(r"</?(?:thinking|think|final|tool_call)[^>]*>", "", result, flags=re.IGNORECASE)
        return result.strip()
    result = re.sub(r"</?(?:thinking|think|final|tool_call)[^>]*>", "", answer, flags=re.IGNORECASE)
    return result.strip()

num_text_samples = len(multimodal_samples) // 3
print(f">> Loading text reasoning samples from Sarvam-105b-Distill-100k (simple_qa configuration), target = {num_text_samples}...")

# Use 'simple_qa' config which contains 95.9k rows with flat keys
ds_text = load_dataset('abhinav0231/Sarvam-105b-Distill-100k', 'simple_qa', split='train', streaming=True)

# Strict clean-up filter pattern (to remove items referencing missing documents or attachments)
corrupt_keywords = [
    "provided contract", "provided agreement", "attached file", "attached document",
    "refer to the table below", "in the table below", "refer to the document",
    "provided bcci", "bcci player agreement", "attached sheet"
]
corrupt_pat = re.compile("|".join(corrupt_keywords), re.IGNORECASE)

text_samples = []
scanned_count = 0

for item in ds_text:
    scanned_count += 1
    if scanned_count % 2000 == 0:
        print(f"  Scanned {scanned_count} text items | Collected: {len(text_samples)} / {num_text_samples}...")

    question = item['question']

    # Skip questions referencing missing context
    if corrupt_pat.search(question) or corrupt_pat.search(item['thinking']) or corrupt_pat.search(item['answer']):
        continue

    # Include all valid multi-domain technical reasoning samples (circuits, physics, thermodynamics, SQL, op-amps, math, etc.)
    system_text = item.get('system_prompt', "You are a helpful assistant. Always reason step-by-step inside <think></think> tags, then write your final answer inside <answer></answer> tags.")
    think = item['thinking']
    clean_answer = clean_sarvam_answer(item['answer'])
    full_response = f"<think>\n{think}\n</think>\n<answer>\n{clean_answer}\n</answer>"

    text_samples.append({
        "image": None,
        "prompt": f"<|im_start|>system\n{system_text}<|im_end|>\n<|im_start|>user\n{question}<|im_end|>\n<|im_start|>assistant\n",
        "response": full_response
    })
    if len(text_samples) >= num_text_samples:
        break

print(f"Collected {len(text_samples)} cleaned multi-domain text reasoning samples.")

>> Loading text reasoning samples from Sarvam-105b-Distill-100k (simple_qa configuration), target = 16000...


README.md:   0%|          | 0.00/2.99k [00:00<?, ?B/s]

  Scanned 2000 text items | Collected: 1998 / 16000...
  Scanned 4000 text items | Collected: 3997 / 16000...
  Scanned 6000 text items | Collected: 5996 / 16000...
  Scanned 8000 text items | Collected: 7995 / 16000...
  Scanned 10000 text items | Collected: 9995 / 16000...
  Scanned 12000 text items | Collected: 11994 / 16000...
  Scanned 14000 text items | Collected: 13994 / 16000...
  Scanned 16000 text items | Collected: 15994 / 16000...
Collected 16000 cleaned multi-domain text reasoning samples.


In [6]:
# 5. Package and Push to Hugging Face Hub
all_samples = multimodal_samples + text_samples
random.seed(SEED)
random.shuffle(all_samples)
print(f"Total mixed dataset size: {len(all_samples)}")

# Build the HF Dataset
features = Features({
    "image": DImage(),
    "prompt": Value("string"),
    "response": Value("string")
})

print(">> Building Hugging Face Dataset...")
dataset = Dataset.from_list(all_samples, features=features)
print(dataset)

print(f">> Pushing dataset to Hugging Face Hub: {TARGET_REPO}...")
dataset.push_to_hub(TARGET_REPO, private=False, token=HF_TOKEN)
print(">> Dataset pushed successfully!")

Total mixed dataset size: 64000
>> Building Hugging Face Dataset...
Dataset({
    features: ['image', 'prompt', 'response'],
    num_rows: 64000
})
>> Pushing dataset to Hugging Face Hub: abhinav0231/Lily-Vision-SFT-Dataset...


Uploading the dataset shards:   0%|          | 0/58 [00:00<?, ? shards/s]

Map:   0%|          | 0/1104 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/12 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :  11%|#         | 79.7MB /  741MB            

README.md:   0%|          | 0.00/363 [00:00<?, ?B/s]

>> Dataset pushed successfully!
